In [1]:
from steer_core.Data.DataManager import DataManager
import steer_opencell_design as ocd

In [2]:
# User Inputs

####################

table_name = 'cell_references' #### change this to cell_teardowns for teardowns
cell_name = "LFP Stacked Prismatic Cell"

#####################

In [3]:
# set some standard materials

conductive_additive = ocd.ConductiveAdditive.from_database("Super P")
binder = ocd.Binder.from_database("PVDF")
insulation = ocd.InsulationMaterial.from_database("Aluminium Oxide, 95%")
separator_material = ocd.SeparatorMaterial.from_database('Polyethylene')
tape_material = ocd.TapeMaterial.from_database("Kapton")
prismatic_material = ocd.PrismaticContainerMaterial.from_database("Steel")

In [4]:
# Create the cathode

cathode_current_collector_material = ocd.CurrentCollectorMaterial.from_database('Aluminum')

cathode_current_collector=ocd.PunchedCurrentCollector(
    material=cathode_current_collector_material,
    width=300,
    height=280,
    tab_height=30,
    tab_position=70,
    tab_width=80,
    thickness=10,
    insulation_width=2
)

cathode_active_material = ocd.CathodeMaterial.from_database("LFP")

cathode_formulation = ocd.CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = ocd.Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=3.1,
    mass_loading=18,
    insulation_material=insulation,
    insulation_thickness=3
)


In [5]:
# Create the anode

cathode_current_collector_material = ocd.CurrentCollectorMaterial.from_database("Copper")

anode_current_collector = ocd.PunchedCurrentCollector(
    material=cathode_current_collector_material,
    width=302,
    height=280,
    tab_height=30,
    tab_position=230,
    tab_width=80,
    thickness=10
)

anode_active_material = ocd.AnodeMaterial.from_database("Synthetic Graphite")

anode_formulation = ocd.AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = ocd.Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=9
)

In [6]:
# create the layup 

separator = ocd.Separator(
    material=separator_material, 
    thickness=12,
    width = 290,
)

my_layup = ocd.ZFoldMonoLayer(
    cathode=my_cathode,
    anode=my_anode,
    separator=separator,
)

my_layup.get_top_down_view().show()


In [7]:
# create the stack assembly

my_stack = ocd.ZFoldStack(
    layup=my_layup,
    n_layers=40,
    additional_separator_wraps=3
)

# looks best in safari
my_stack.get_side_view().show()

In [8]:
# make the electrolyte

my_electrolyte = ocd.Electrolyte(
    name="1M NaPF6 in EC:PC:DMC (1:1:1 wt%)",
    density=1.2,
    specific_cost=10,
    color="#FF9D00"
)


In [9]:
# make the encapsulation

cathode_terminal_connector = ocd.PrismaticTerminalConnector(
    material=prismatic_material,
    thickness=2,
    width=40,
    length=40
)

anode_terminal_connector = ocd.PrismaticTerminalConnector(
    material=prismatic_material,
    thickness=2,
    width=40,
    length=40
)

lid_assembly = ocd.PrismaticLidAssembly(
    material=prismatic_material,
    thickness=8,
)

canister = ocd.PrismaticCanister(
    material=prismatic_material,
    width=310,
    length=52,
    height=300,
    wall_thickness=1
)

my_encapsulation = ocd.PrismaticEncapsulation(
    canister=canister,
    cathode_terminal_connector=cathode_terminal_connector,
    anode_terminal_connector=anode_terminal_connector,
    lid_assembly=lid_assembly,
)

In [10]:
# make the cell

cell = ocd.PrismaticCell(
    reference_electrode_assembly=my_stack,
    electrolyte=my_electrolyte,
    electrolyte_overfill=0.1,
    encapsulation=my_encapsulation,
    n_electrode_assembly=4,
    clipped_tab_length=4,
    name=cell_name
)

# looks better in safari
cell.get_side_view().show()
cell.get_top_down_view().show()

In [11]:
cell.plot_mass_breakdown(width=800, height=800)

In [12]:
cell.plot_cost_breakdown(width=800, height=800)

In [13]:
cell.get_capacity_plot(width=1300, height=800).show()

In [14]:
print(f"Cost ($): {cell.cost}")
print(f"Mass (g): {cell.mass}")
print(f"Energy Density (Wh/L): {cell.volumetric_energy}")
print(f"Energy (Wh): {cell.energy}")
print(f"Energy Density (Wh/kg): {cell.specific_energy}")
print(f"Normalized Cost ($/kWh): {cell.cost_per_energy}")

Cost ($): 91.17
Mass (g): 13531.32
Energy Density (Wh/L): 436.03
Energy (Wh): 2108.62
Energy Density (Wh/kg): 155.83
Normalized Cost ($/kWh): 43.24


In [15]:
import pandas as pd
import datetime as dt
import re
from steer_opencell_design import __version__


db = DataManager()

db.remove_data(table_name=table_name, condition=f"name = '{cell.name}'")

# insert the cell into the database
db.insert_data(table_name=table_name, data=pd.DataFrame({
    'name': [cell.name],
    'object': [cell.serialize()],
    'form_factor': [cell.form_factor],
    'internal_construction': [cell.internal_construction],
    'date_created': [dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'chemistry': [cell.reference_chemistry]
}))

db.get_data(table_name)

,name,object,form_factor,internal_construction,date_created,version,chemistry
0,NFM111 Cylindrical Tabless Cell,b'\x01x\x9c\xac\xd7\x07<\x96\xed\xdf0p\\\x03-\...,Cylindrical Cell,Wound Jelly Roll,2026-01-20 12:28:19,1.0.2,Na/Na+
1,NFM111 Z-Fold Prismatic Cell,b'\x01x\x9c\x9c\xd6w \x96\xfd\xdb8~\\\x03\x95\...,Prismatic Cell,ZFold Stack,2026-01-20 12:28:23,1.0.2,Na/Na+
2,NFPP Flat Wound Prismatic Cell,b'\x01x\x9c\xec\xfdy\\\ra\xff\xc7\x8f\xb7o\xda...,Prismatic Cell,Flat Wound Jelly Roll,2026-01-20 12:28:28,1.0.2,Na/Na+
3,NMC811 Cylindrical Tabbed Cell,b'\x01x\x9c\xac\xbd\x05X\x95K\xf7\xf7\x0fHK7\x...,Cylindrical Cell,Wound Jelly Roll,2026-01-20 12:28:34,1.0.2,Li/Li+
4,NMC811 Stacked Pouch Cell,b'\x01x\x9c\x9c\x9d\x07XU\xc9\xb2\xb6\x01\xc9\...,Pouch Cell,ZFold Stack,2026-01-20 12:28:38,1.0.2,Li/Li+
5,NMC811 Stacked Prismatic Cell,b'\x01x\x9c\x9c\xbd\x05XU[\xd7\xf7\rHK7\x1b6\x...,Prismatic Cell,ZFold Stack,2026-01-20 12:28:44,1.0.2,Li/Li+
6,LFP Cylindrical Tabless Cell,b'\x01x\x9c\xec\xddw\\\xce_\xff8\xf0\xd0\xde[Q...,Cylindrical Cell,Wound Jelly Roll,2026-01-20 12:29:14,1.0.2,Li/Li+
7,LFP Flat Wound Prismatic Cell,b'\x01x\x9c\xec\xddw\\\xcd\xef\xff8\xfe\xa2\xb...,Prismatic Cell,Flat Wound Jelly Roll,2026-01-20 12:29:36,1.0.2,Li/Li+
8,LFP Stacked Prismatic Cell,b'\x01x\x9c\xec\xddy\\M\xdf\xfe8\xfeR\xd2<\x8f...,Prismatic Cell,ZFold Stack,2026-01-20 12:29:48,1.0.2,Li/Li+


In [16]:
size_mb = len(cell.serialize()) / (1024 ** 2)
print(f"{size_mb:.2f} MB")

4.10 MB
